# 08 — LangGraph State (`graph/state.py`)
`AgentState` is a `TypedDict` — the single shared state object that flows through the entire graph.

Key design decisions:
- `agent_results`, `sources`, `anomalies`, `errors`, `conversation_history` use `Annotated[List, operator.add]` — meaning multiple parallel nodes can write to them and they **accumulate** (not overwrite)
- `initial_state()` is a factory function that creates a clean starting state


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'codebase', 'codebase', 'src'))
os.makedirs("logs", exist_ok=True)

## 1. initial_state() Factory

In [ ]:
from graph.state import initial_state, AgentState

state = initial_state(
    query="What is GRR for retention last month?",
    thread_id="thread-001",
    user_id="analyst@company.com",
    time_range="last_month",
)
print("State fields:")
for k, v in state.items():
    print(f"  {k:<24}: {repr(v)}")

## 2. State Fields — Input Group

In [ ]:
# Fields set at query time
input_fields = ["query", "thread_id", "user_id", "time_range", "data_products"]
for f in input_fields:
    print(f"{f:<20}: {state[f]}")

## 3. State Fields — Routing Group (set by supervisor_node)

In [ ]:
# Simulate what supervisor_node would set
state["intent"] = "metric_analysis"
state["next_agents"] = ["information", "knowledge"]
state["data_products"] = ["retention", "cac"]
state["confidence"] = 0.91

routing_fields = ["intent", "next_agents", "data_products", "confidence"]
for f in routing_fields:
    print(f"{f:<20}: {state[f]}")

## 4. Accumulating Lists (operator.add semantics)

In [ ]:
import operator

# In LangGraph, agent_results uses Annotated[List[dict], operator.add]
# This means updates ACCUMULATE across parallel nodes — not overwrite

result_from_info_node   = [{"agent": "information_agent", "success": True, "summary": "GRR=87.5%"}]
result_from_knowledge   = [{"agent": "knowledge_agent",   "success": True, "summary": "GRR defined as..."}]

# How LangGraph merges them:
merged = operator.add(result_from_info_node, result_from_knowledge)
print("Accumulated agent_results:")
for r in merged:
    print(f"  {r}")

## 5. State Fields — Write Actions (HITL)

In [ ]:
# Simulate HITL flow
state["anomalies"] = ["⚠️ GRR (82%) below 85% threshold"]
state["pending_action"] = {
    "action": "create_jira_tickets",
    "anomalies": ["⚠️ GRR (82%) below 85% threshold"],
    "products": ["retention"],
    "count": 1,
    "message": "Found 1 critical anomaly. Approve Jira ticket creation?",
}
state["approved"] = False

print("pending_action:", state["pending_action"]["message"])
print("approved      :", state["approved"])
print("anomalies     :", state["anomalies"])

## 6. State Fields — Memory & Metadata

In [ ]:
# After a run completes
state["query_id"]      = "a1b2c3d4"
state["execution_ms"]  = 342.5
state["final_summary"] = "GRR is 87.5%, above the 85% threshold..."
state["start_time"]    = 1234567890.0
state["guardrail_passed"] = True

print("query_id        :", state["query_id"])
print("execution_ms    :", state["execution_ms"])
print("guardrail_passed:", state["guardrail_passed"])
print("final_summary   :", state["final_summary"][:60], "...")

## 7. Full State Keys Reference

In [ ]:
import typing
print("All AgentState fields:")
for field_name, field_type in AgentState.__annotations__.items():
    print(f"  {field_name:<28}: {str(field_type)[:50]}")